### Build a Question Answering application over a Graph Database

In [7]:
NEO4J_URI= "neo4j+s://aa85a2a9.databases.neo4j.io"
NEO4J_USERNAME= "neo4j"
NEO4J_PASSWORD= "wvL0RkKbNxm6x5eZzKVFxWByH6iZiAq9Da5QHNchqqo"

In [8]:
from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(url= NEO4J_URI, username= NEO4J_USERNAME, password = NEO4J_PASSWORD)
graph

# creating a connection between your LangChain application and a Neo4j graph database.

# Neo4jGraph= initializes a graph database client.
# url =  specifies the Neo4j database address.
# username and password are used for authentication.
# The resulting graph object lets you execute Cypher queries, inspect the schema, and interact with the Neo4j database through LangChain.

In [10]:
## Dataset Moview 
moview_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))


"""

In [11]:
graph.query(moview_query)

[]

In [12]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [13]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()

In [ ]:
from langchain_classic.chains import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(graph = graph, llm = llm , allow_dangerous_requests=True, verbose = True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x10880f640>, cypher_generation_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x11e900400>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11fdb3130>, ro

In [19]:
response=chain.invoke({"query":"Who was the director of the movie Casino"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Casino"})
RETURN p.name


[#E851]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-aa85a2a9-599b-0001.production-orch-0337.neo4j.io', 7687)) (ResolvedIPv4Address(('34.142.144.246', 7687))): OSError('No data')
Transaction failed and will be retried in 0.9700612745122333s (Failed to read from defunct connection IPv4Address(('p-aa85a2a9-599b-0001.production-orch-0337.neo4j.io', 7687)) (ResolvedIPv4Address(('34.142.144.246', 7687))))


Full Context:
[{'p.name': 'Martin Scorsese'}]

> Finished chain.


{'query': 'Who was the director of the movie Casino',
 'result': 'Martin Scorsese'}

In [20]:
response=chain.invoke({"query":"How many movies has Tom Hanks acted in"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (:Person {name: "Tom Hanks"})-[:ACTED_IN]->(movie:Movie)
RETURN COUNT(movie) as numberOfMoviesTomHanksActedIn;
Full Context:
[{'numberOfMoviesTomHanksActedIn': 2}]

> Finished chain.


{'query': 'How many movies has Tom Hanks acted in',
 'result': 'Tom Hanks has acted in 2 movies.'}